In [1]:
import pandas as pd
import os
import subprocess
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from pywinauto.application import Application
from pywinauto.keyboard import send_keys
from pywinauto import Desktop
import re
import pyperclip
import time

In [103]:
with open("stories_text_BRF\\02_Aqua.txt", "r") as f:
    txt = f.read()
pyperclip.copy(txt)
print(os.getcwd())

c:\Users\PC\GitHub\naturalstories


In [ ]:
brf_dir = "stories_text_BRF/"
tfiles = [ff for ff in os.listdir(brf_dir) if (ff.split(".")[-1]=="txt") and ff!="01_Boar.txt"]
overwrite = False

for ff in tfiles:
    brf_path = Path.cwd() / "stories_text_BRF" / f"{ff.split(".")[0]}.brf"
    if not overwrite and brf_path.exists():
        continue
    # print(ff)
    # continue
    app = Application(backend="uia").start("brailleblaster.exe")
    time.sleep(5)
    # win = app.window(title_re=".*BrailleBlaster.*")
    # win.wait("visible", timeout=20)
    # win.set_focus()
    send_keys("^o")
    time.sleep(1)


    # inspect once if needed
    # dlg.print_control_identifiers()

    # filename_box = dlg.child_window(title="File name:", control_type="Edit")
    file_path = Path.cwd() / "stories_text_BRF" / "01_Boar.bbz"
    pyperclip.copy(str(file_path))
    send_keys("^v")
    time.sleep(0.3)
    send_keys("{ENTER}")
    time.sleep(1)


    send_keys("^{HOME}")       # Ctrl+Home
    time.sleep(1)

    send_keys("+{END}")       # Ctrl+Shift+End
    time.sleep(1)

    for ii in range(210):
        send_keys("+{DOWN}")

    send_keys("{DEL}")
    time.sleep(1)

    with open(f"stories_text_BRF\\{ff}", "r") as f:
        txt = f.read()
    pyperclip.copy(txt)

    # send_keys("^a")
    send_keys("^v")
    time.sleep(1)
    send_keys("%fe{RIGHT}v{ENTER}")
    time.sleep(1)

    # remove brf_path if it exists
    if brf_path.exists():
        brf_path.unlink()
    pyperclip.copy(str(brf_path))
    send_keys("^v")
    time.sleep(1)
    send_keys("{ENTER}")
    time.sleep(1)

    send_keys("%{F4}")
    send_keys("n")
    send_keys("{ENTER}")
    time.sleep(5)

In [34]:
# ======= image creation params =======
DPI = 300

CARD_W_IN = 8
CARD_H_IN = 5.5

BRAILLE_X_IN = 1.75
BRAILLE_Y_IN = 0.25
BRAILLE_W_IN = 6
BRAILLE_H_IN = 5

BORDER_W_IN = 0.08

SWELL_BRAILLE_FONT = r"swell-braille.ttf"
ID_FONT = r"C:\\Windows\\Fonts\\impact.ttf"
TEXT_FONT = r"C:\\Windows\\Fonts\\calibri.ttf"


braille_font = ImageFont.truetype(SWELL_BRAILLE_FONT, size=100)
id_font = ImageFont.truetype(ID_FONT, size=150)
back_font = ImageFont.truetype(TEXT_FONT, size=100)

def inch(x):
    return int(round(x * DPI))

def render_front_card(brf_text, card_id, out_path):
    img = Image.new("RGB", (inch(CARD_W_IN), inch(CARD_H_IN)), "white")
    draw = ImageDraw.Draw(img)

    x0 = inch(BRAILLE_X_IN)
    y0 = inch(BRAILLE_Y_IN)
    x1 = x0 + inch(BRAILLE_W_IN)
    y1 = y0 + inch(BRAILLE_H_IN)

    # Red border
    draw.rectangle(
        [x0, y0, x1, y1],
        outline=(220, 0, 0),
        width=inch(BORDER_W_IN)
    )

    # Braille text
    pad_x = inch(0.12)
    pad_y = inch(0.12)

    draw.multiline_text(
        (x0 + pad_x, y0 + pad_y),
        brf_text,
        font=braille_font,
        fill="black",
        spacing=40,
    )

    # Card ID, lower-left corner
    draw.text(
        (inch(0.25), inch(4.75)),
        card_id,
        font=id_font,
        fill="black",
        va="bottom"
    )

    img.save(out_path, quality=95)
    return

def wrap_text_by_pixels(text, font, max_width_px, draw):
    words = text.replace("\n", " ").split()
    lines = []
    current = ""

    for word in words:
        test = word if not current else current + " " + word
        bbox = draw.textbbox((0, 0), test, font=font)
        if bbox[2] - bbox[0] <= max_width_px:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return "\n".join(lines)

def render_back_card(txt, card_id, out_path):
    img = Image.new("RGB", (inch(CARD_W_IN), inch(CARD_H_IN)), "white")
    draw = ImageDraw.Draw(img)

    # Card ID / serial
    draw.text(
        (inch(0.25), inch(0.25)),
        card_id,
        font=id_font,
        fill="black"
    )

    # Text snippet
    text_x = inch(0.75)
    text_y = inch(1.0)
    text_w = inch(6.75)

    wrapped = wrap_text_by_pixels(txt, back_font, text_w, draw)

    draw.multiline_text(
        (text_x, text_y),
        wrapped,
        font=back_font,
        fill="black",
        spacing=30,
    )

    img.save(out_path, quality=95)
    return


In [47]:
brf_dir = "stories_text_BRF/"
bfiles = [ff for ff in os.listdir(brf_dir) if ff.split(".")[-1]=="brf"]
brf_df = pd.DataFrame(columns=["story", "page", "brf", "txt"])
translator = ".\\liblouis-3.37.0-win64\\bin\\lou_translate.exe"

OUT_DIR = Path("card_images")
FRONT_DIR = OUT_DIR / "front"
BACK_DIR = OUT_DIR / "back"
# FRONT_DIR.mkdir(parents=True, exist_ok=True)
# BACK_DIR.mkdir(parents=True, exist_ok=True)

cwd = os.getcwd()

bb_to_louis = {"\\":"|", "[":"{", "]":"}", "^":"~"}

for ff in bfiles[1:]:
    story_id = ff.split("_")[0]
    brf_text = open(brf_dir+ff, "r").read()

    # print(len(brf_text.split("\f")))
    # continue
    # Each line in each page is indicated by a new line. 
    # Pages are separated by an additional new line.
    # Empty lines in the last page are still marked by newlines, so we have to strip them.
    pages = brf_text.rstrip().lower().split("\f")
    print(len(pages))
    for ii,pp in enumerate(pages):
        for kk,vv in bb_to_louis.items():
            pp = pp.replace(kk,vv)
        print(pp)
        print("========================")
        with open("tmp.txt", "w+") as tmp:
            tmp.write(pp)
        cmd = f"{translator} -b en-us-g2.ctb < tmp.txt"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True,cwd=cwd)
        txt = result.stdout

        card_id = f"{story_id}-{(ii+1):02d}"
        front_path = FRONT_DIR / f"{card_id}_front.jpg"
        back_path = BACK_DIR / f"{card_id}_back.jpg"
        render_front_card(pp, card_id, front_path)
        render_back_card(txt, card_id, back_path)
        # print(cmd)
        print("\n\n")


16
  ,a cle> & joy|s "d x 0
& | on ! wide1 op5 sea
?|s&s ~u ?|s&s ( sp>kl+
wat}-drops excit$ by
gett+ to play 9 ! oc1n
d.ed all >.d4 ,"o ( ~! 0
a m}ry ll fell{ "nd
,aqua :o d.ed on ! silv}
backs ( ! fi%es z !y
plung$ up & d{n 9 !
waves1 &1 no matt} h{




hi< he sprung1 he alw
came d{n ag 9to 8 "m's
lap4 ,8 "m1 y "k1 0 !
,oc1n1 & v b1uti;l %e
look$ t summ} "d 9 h}
d>k blue dress & :ite
ru6les4 ,by & by1 !
happy wat}-drop tir$ ( 8
play1 & look+ up to !
cle> sky abv hm1 ?"| he
wd l to h a sail on "o (




! :ite1 m>%mall{-l
cl|ds1 9/1d ( sp5d+ 8
:ole life 9 ! oc1n4 ,( !
sky1 ll ,aqua _h alw be5
afraid1 b he decid$ to
f9ally face 8 demons &
su7e/$ a solu;n 9volv+ !
,sun c>ry+ hm up to !
sky wd 2 id1l4 ,! ,sun
"u/ood ,aqua's reque/
came f 8 he>t1 s he




acquiesc$ & al took "e s
_m o!r drops1 s t ,aqua
mi<t n 2 l"o"s on ! way4
,x 0 only ! sun t knew
?1 h{"e1 = all ! o!r
drops _h be5 *ang$ 9to
f9e mi/ or vapor & ,aqua
cd n see !m4 ,d y "k :at
vapor is8 ,if y br1! 9to
! air1 :5 x is cold 5|

  ,IF Y 7 TO J\RNEY TO !
,NOR? ( ,5GL&1 Y WD COME
TO A VALLEY T IS SURR.D$
BY MOORS Z HI< Z M.TA9S4
,X IS 9 ? VALLEY ": Y WD
F9D ! C;Y ( ,BRAD=D1 ":
ONCE A ?\S& SP9N+-J5NIES
T HUMM$ & CLATT]$ SPUN
WOOL 9TO M"OY = !
L;G-BE>D$ MILL [N]S4 ,T
ALL MILL [N]S 7 G5]ALLY

BUSY Z B1V]S & Q PL1S$ )
!MVS = 2+ S SU3ESS;L &
WELL (F 0 "KN TO !
RESID5TS ( ,BRAD=D1 & IF
Y 7 TO G 9TO ! C;Y TO
VISIT ! /ATELY ,C;Y
,HALL1 Y WD SEE "! !
,CRE/ ( ! ,C;Y (
,BRAD=D1 : ^? SAME
MILL-[N]S CR1T$ TO
CELEBRATE _! A*IEVE;TS4

,X %[S A S9I/] LOOK+
BO>'S H1D SITT+ ON TOP (
A WELL : SEEMS PUZZL+ AT
F/1 B ! R1SON = ? SYMBOL
IS A MATT] ( LEG5D4
  ,"! 0 ONCE1 LEG5D HAS
X1 A FE>"S BO>1 : LIV$ 9
A WOOD LOCAT$ J \TSIDE !
MANOR ( ,BRAD=D4 ,A
S\RCE ( GRT TR\BLE TO !
LOCAL FOLK ! BO> WAS1

BR++ T]ROR TO ! P1CE;L
FLOCKS & RAVAG+ !
C.TRYSIDE >.D4 ,EV5
WORSE1 H["E1 ! BO> MO/
LIK$ TO G TO ! WELL T 0
9 ! WOOD & DR9K XS FRE%
WAT]1 S T ! P ( ,BRAD=D
_H SECOND ?"\S AB VISIT+
! WELL4
  ,T ! P ( ,BRAD=D BORE
! BRUNT ( ! B1/'S F]OC;Y

0 UNF